In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Load data

import os

housing = pd.read_csv('housing.csv')

In [ ]:
housing.head(3)

In [ ]:
housing.info()

## Data understanding

There are 20,640 instances in the dataset, which means that it is fairly small by
Machine Learning standards, but it?s perfect to get started. Notice that the total_bed
rooms attribute has only 20,433 non-null values, meaning that 207 districts are miss?
ing this feature. We will need to take care of this later.

In [ ]:
housing['ocean_proximity'].value_counts()

In [ ]:
housing.describe()

In [ ]:
# Another quick way is visualize the data with graph

%matplotlib inline 

import matplotlib.pyplot as plt

housing.hist(bins=50, figsize=(20, 15))

plt.show()

# Some problems in  histogram

* Different scale of attributes
* Tail very heavy => transformation to bell shape
* Data is limited

In [ ]:
housing.plot(kind='scatter', x='longitude', y='latitude', alpha=0.1)

In [ ]:
# Find the relate between housing price and location

housing.plot(kind='scatter', x='longitude', y='latitude', alpha=0.5, 
            s=housing['population'] / 50, label='population', figsize=(12, 8),
            c='median_house_value', cmap=plt.get_cmap('jet'), colorbar=True)


# Map of california

![](https://www.google.com.vn/maps/vt/data=ILHFXe3jioD0HIi5Q_vMk7fEl4SNBFTkdoIwrlaX36l4m0BgMp2xJ9s9uER_ZQwjwvJzGsK5wOrvFd6-nxgbgeKi3jAlNDkSOoHLlmaLLoRyqWN76HdEAZdoiqrPNP1tlaVFl5Zw6eQrixmFnx9fAshe8MrPbUpCUDxmzkSeQ35_d3WTCmdekTu-JrcDw3Bl-pARLNx4wmddm097Xih0qfnzequqSfUW8r2Jsw0PGQJTmQ)

# Compute the correlation matrix between attributes

In [ ]:
corr_matrix = housing.corr()

In [ ]:
corr_matrix['median_house_value'].sort_values()

# Cleaning data

In [ ]:
median = housing["total_bedrooms"].median()

housing["total_bedrooms"].fillna(median, inplace=True)

# Attributes combination



In [ ]:
housing['room_per_household'] = housing['total_rooms'] / housing['households']

housing['bedroom_per_household'] = housing['total_bedrooms'] / housing['households']

housing['population_per_household'] = housing['population'] / housing['households']

In [ ]:
housing.head(3)

In [ ]:
corr_matrix = housing.corr()

corr_matrix['median_house_value'].sort_values()

# Handing the text to categorical attributes

In [ ]:
# Pandas get dumies method

housing_dummies = pd.get_dummies(housing, prefix=['ocean_proximity'])
housing_dummies.head(3)

# Feature scaling


* **Normalization** also call with min-max scaling to put the number value in range. We do this by subtract?
ing the min value and dividing by the max minus the min
* **Standardization** first it subtracts the mean value (so standardized
values always have a zero mean), and then it divides by the variance so that the result?
ing distribution has unit variance.

In [ ]:
import numpy as np

tmp = np.array([1, 2, 3, 4, 4, 5, 67, 2, 21, 54])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()

min_max_scaler.fit(tmp.reshape(-1, 1))

min_max_scaler.transform([[1, 3, 2]])

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()

std_scaler.fit(tmp.reshape(-1, 1))

std_scaler.transform([[1, 3, 2]])

In [ ]:
(2 - tmp.mean()) / (tmp.std())

In [ ]:
# Transform data

scaler = MinMaxScaler()

housing_features = housing_dummies.drop('median_house_value', axis=1)

X_data = scaler.fit_transform(housing_features)

In [ ]:
y_data = np.array(housing_dummies['median_house_value'])

y_data = y_data.reshape(len(y_data), 1)

## Note

Unlike min-max scaling, standardization does not
bound values to a specific range, which may be a problem for some algorithms (e.g.,
neural networks often expect an input value ranging from 0 to 1). However, standard?
ization is much less affected by outliers

In [ ]:
# Train test split 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
y_test.shape

In [ ]:
X_train[0]

In [ ]:
# Placeholder for pass data
import tensorflow as tf

n_dim = 16

X = tf.placeholder(tf.float32, [None, n_dim])

Y = tf.placeholder(tf.float32, [None, 1])

In [ ]:
# Trainable weights

W = tf.Variable(tf.random_uniform([n_dim, 1]), dtype=tf.float32)

b = tf.Variable(tf.zeros(1, dtype = tf.float32))

In [ ]:
pred = tf.add(tf.matmul(X, W), b)

mse_loss = tf.reduce_mean(tf.square(Y - pred))

loss = tf.sqrt(mse_loss)

learning_rate = 300

optimizer = tf.train.GradientDescentOptimizer(learning_rate=learning_rate).minimize(loss)

In [ ]:
init = tf.global_variables_initializer()

In [ ]:
sess = tf.Session()

In [ ]:
# Training

sess.run(init)

In [ ]:
epochs = 50000
for epoch in range(epochs):
    
    sess.run(optimizer, feed_dict={X: X_train, Y: y_train})
    
    train_loss = sess.run(loss, feed_dict={X : X_train, Y: y_train})
    test_loss = sess.run(loss, feed_dict={X : X_test, Y: y_test})
 
    if epoch % 1000 == 0:
        print("Epoch {} Train loss {} Test loss = {}".format(epoch, train_loss, test_loss))
        
print("Training finished")

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

In [ ]:
lin_reg.coef_.reshape(n_dim, 1)

In [ ]:
y_pred = lin_reg.predict(X_test)

In [ ]:
y_model_pred = sess.run(pred, feed_dict={X : X_test, Y: y_test})

In [ ]:
from sklearn.metrics import mean_squared_error

def rmse_evaluate(y_pred, y_test):
    lin_mse = mean_squared_error(y_pred, y_test)

    return np.sqrt(lin_mse)

In [ ]:
rmse_evaluate(y_pred, y_test)

In [ ]:
rmse_evaluate(y_model_pred, y_test)